# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

In [1]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU is available: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected. For faster training, consider running this notebook on Google Colab with a GPU runtime.")
    print("On Google Colab, go to: Runtime > Change runtime type > Hardware accelerator > GPU")

✅ GPU is available: Tesla T4


Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [2]:
import os
print(os.getcwd())   # prints current working directory
print(os.listdir())        # files in current directory


/content
['.config', 'sample_data']


In [5]:
import pandas as pd

# 1. Read dataset
df = pd.read_csv("/content/tic-tac-toe.csv")

# 2. Inspect
print(df.shape)
display(df.head())
df.info()

print("\nMissing values:")
print(df.isnull().sum())


(958, 10)


,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,x,x,x,x,o,o,x,o,o,True
1,x,x,x,x,o,o,o,x,o,True
2,x,x,x,x,o,o,o,o,x,True
3,x,x,x,x,o,o,o,b,b,True
4,x,x,x,x,o,o,b,o,b,True


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   TL      958 non-null    object
 1   TM      958 non-null    object
 2   TR      958 non-null    object
 3   ML      958 non-null    object
 4   MM      958 non-null    object
 5   MR      958 non-null    object
 6   BL      958 non-null    object
 7   BM      958 non-null    object
 8   BR      958 non-null    object
 9   class   958 non-null    bool  
dtypes: bool(1), object(9)
memory usage: 68.4+ KB

Missing values:
TL       0
TM       0
TR       0
ML       0
MM       0
MR       0
BL       0
BM       0
BR       0
class    0
dtype: int64


In [6]:
display(df.columns.tolist())
print(df['TL'].unique())
print(df['TM'].unique())
print(df['TR'].unique())
print(df['ML'].unique())
print(df['MM'].unique())
print(df['MR'].unique())
print(df['TR'].unique())
print(df['BL'].unique())
print(df['BM'].unique())
print(df['BR'].unique())

#Convert the categorical values to numeric in all columns.

mapping = {
    "x": 1,
    "o": -1,
    "b": 0
}

df_numeric = df.copy()

for column in df_numeric.columns[:-1]:
    df_numeric[column] = df_numeric[column].map(mapping)

df_numeric.sample(5)
df_numeric.shape
df_numeric.dtypes

['TL', 'TM', 'TR', 'ML', 'MM', 'MR', 'BL', 'BM', 'BR', 'class']

['x' 'o' 'b']
['x' 'o' 'b']
['x' 'o' 'b']
['x' 'o' 'b']
['o' 'b' 'x']
['o' 'b' 'x']
['x' 'o' 'b']
['x' 'o' 'b']
['o' 'x' 'b']
['o' 'x' 'b']


,0
TL,int64
TM,int64
TR,int64
ML,int64
MM,int64
MR,int64
BL,int64
BM,int64
BR,int64
class,bool


In [7]:
# 4. Separate inputs and output

X = df_numeric.iloc[:, :-1]

y = df_numeric.iloc[:, -1]
y = y.astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)
print(y.unique())

# Normalize the input data.
#The categorical board positions were encoded as -1, 0, and 1. Since all nine input features already have the same small numerical range, additional normalization/standardization is not necessary.

X shape: (958, 9)
y shape: (958,)
[1 0]


## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [8]:
# 1.Split the training and test data.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


#Create a `Sequential` model and adding activation functions.

import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(9,)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(2, activation="softmax")
])

X_train: (766, 9)
X_test: (192, 9)
y_train: (766,)
y_test: (192,)


In [10]:
#Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.


model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

#Fit the training data.
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    validation_split=0.2,
    verbose=1
)

#Evaluate your neural network model with the test data.
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

model.save("tic-tac-toe.keras")

Epoch 1/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step - accuracy: 0.9755 - loss: 0.1488 - val_accuracy: 0.9481 - val_loss: 0.1935
Epoch 2/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9771 - loss: 0.1338 - val_accuracy: 0.9610 - val_loss: 0.1793
Epoch 3/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9804 - loss: 0.1228 - val_accuracy: 0.9481 - val_loss: 0.1696
Epoch 4/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9804 - loss: 0.1139 - val_accuracy: 0.9675 - val_loss: 0.1593
Epoch 5/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9837 - loss: 0.1050 - val_accuracy: 0.9545 - val_loss: 0.1490
Epoch 6/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9853 - loss: 0.0945 - val_accuracy: 0.9675 - val_loss: 0.1390
Epoch 7/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9853 - loss: 0.0859 - val_accuracy: 0.9740 - val_loss: 0.1298
Epoch 8/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9886 - loss: 0.0785 - val_accuracy: 0.9675 - val_los

## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [12]:
loaded_model = tf.keras.models.load_model("tic-tac-toe.keras")

import numpy as np

indices = np.random.choice(len(X_test), size=5, replace=False)

X_samples = X_test.iloc[indices]
y_actual = y_test.iloc[indices]

predictions = loaded_model.predict(X_samples)

y_pred = np.argmax(predictions, axis=1)

print("Actual:   ", y_actual.to_numpy())
print("Predicted:", y_pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 330ms/step
Actual:    [0 1 1 1 1]
Predicted: [0 1 1 1 1]


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [13]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(9,)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(2, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    validation_split=0.2,
    verbose=1
)

test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

Epoch 1/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 73ms/step - accuracy: 0.5817 - loss: 0.6810 - val_accuracy: 0.6429 - val_loss: 0.6485
Epoch 2/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6732 - loss: 0.6119 - val_accuracy: 0.6558 - val_loss: 0.6088
Epoch 3/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7026 - loss: 0.5706 - val_accuracy: 0.7013 - val_loss: 0.5745
Epoch 4/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7451 - loss: 0.5358 - val_accuracy: 0.7208 - val_loss: 0.5483
Epoch 5/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7810 - loss: 0.5035 - val_accuracy: 0.7532 - val_loss: 0.5241
Epoch 6/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8007 - loss: 0.4735 - val_accuracy: 0.7662 - val_loss: 0.5022
Epoch 7/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8301 - loss: 0.4394 - val_accuracy: 0.7727 - val_loss: 0.4743
Epoch 8/30
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8448 - loss: 0.4074 - val_accuracy: 0.8052 - val_loss

**Which approach(es) did you find helpful to improve your model performance?**

In [ ]:
#Experiment 1:I used a neural network with 32 and 16 neurons in the hidden layers. The model achieved a test loss of 0.055 and test accuracy of 98.44%.

#Experiment 2: I increased the hidden layers to 64 and 32 neurons. The test loss increased slightly to 0.058 and test accuracy decreased to 97.92%.

#Increasing the number of neurons therefore did not improve the model, so I kept the original 32/16 architecture.